# 06 — Chaos Engineering & Fault Injection

**Pipeline:** DistributedTrainingSimulator — node failure, gradient corruption, slow-node, checkpoint/recovery

```
Distributed Training Simulation
        │
        ├── Scenario A: Node failure (random node dies mid-step)
        │       └── recovery: checkpoint reload + gang restart
        │
        ├── Scenario B: Gradient corruption (NaN injection)
        │       └── recovery: gradient clipping + skip step
        │
        ├── Scenario C: Slow node (stragglers in AllReduce)
        │       └── recovery: timeout + node exclusion
        │
        └── Monte Carlo: job survival probability vs fleet size
                    │
                    ▼
             Reliability curve plot
```

**References:**
- Check-N-Run: Eisenman et al. (2022) https://arxiv.org/abs/2401.06172
- Bamboo: Yang et al. (2023) https://arxiv.org/abs/2303.02399

> **CPU-friendly:** This is a pure simulation — no GPU required.

In [ ]:
from __future__ import annotations

import dataclasses
import logging
import math
import os
import random
import time
from dataclasses import dataclass, field
from pathlib import Path

import wandb

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

CFG = {
    "num_nodes": 64,
    "failure_rate_per_hour": 0.01,
    "training_hours": 72,
    "checkpoint_interval_min": 15,
    "monte_carlo_trials": 1000,
    "scenarios": ["node_failure", "gradient_corruption", "slow_node"],
}

if not os.getenv("WANDB_DISABLED"):
    wandb.init(project="ai-transformer-research-hub", config=CFG, job_type="chaos-test")


In [ ]:
# Full DistributedTrainingSimulator implementation
# Check-N-Run: Eisenman et al. (2022) https://arxiv.org/abs/2401.06172
# Bamboo elastic training: Yang et al. (2023) https://arxiv.org/abs/2303.02399


@dataclass
class ScenarioResult:
    """Outcome of a single chaos scenario run."""

    scenario: str
    survived: bool
    lost_progress_min: float
    recovery_time_min: float
    num_failures: int
    final_step: int
    events: list[str] = field(default_factory=list)


class DistributedTrainingSimulator:
    """Discrete-event simulator for distributed deep-learning training chaos.

    Models a fleet of num_nodes workers executing a synchronous data-parallel
    training job. Three failure modes are supported:

    * node_failure: a random worker crashes mid-step; the job resumes after
      reloading from the most recent checkpoint (Check-N-Run strategy;
      Eisenman et al. 2022 https://arxiv.org/abs/2401.06172).
    * gradient_corruption: NaN injected into gradients at a random step;
      recovery skips the corrupted step.
    * slow_node: a straggler causes AllReduce barrier delays; excluded if
      delay exceeds timeout (Bamboo; Yang et al. 2023
      https://arxiv.org/abs/2303.02399).

    The monte_carlo_survival method estimates job-survival probability via
    repeated Bernoulli trials.

    Args:
        num_nodes: Size of the GPU cluster.
        failure_rate: Per-node failure rate in events per hour (Poisson).
        checkpoint_interval_min: Minutes between periodic checkpoints.
        seed: Random seed for reproducibility.
    """

    def __init__(
        self,
        num_nodes: int,
        failure_rate: float,
        checkpoint_interval_min: float,
        seed: int = 42,
    ) -> None:
        self.num_nodes = num_nodes
        self.failure_rate = failure_rate
        self.checkpoint_interval_min = checkpoint_interval_min
        self._rng = random.Random(seed)
        log.info(
            "DistributedTrainingSimulator: %d nodes, lambda=%.4f/hr, ckpt_interval=%g min",
            num_nodes, failure_rate, checkpoint_interval_min,
        )

    # ------------------------------------------------------------------ helpers

    def _time_to_next_failure_min(self) -> float:
        """Sample inter-failure interval (minutes) from a Poisson process."""
        rate_per_min = self.failure_rate * self.num_nodes / 60.0
        if rate_per_min <= 0:
            return float("inf")
        return self._rng.expovariate(rate_per_min)

    def _last_checkpoint_before(self, elapsed_min: float) -> float:
        """Return the elapsed time of the checkpoint just before elapsed_min."""
        interval = self.checkpoint_interval_min
        return math.floor(elapsed_min / interval) * interval

    # ------------------------------------------------------------------ scenarios

    def _run_node_failure(self, training_hours: float) -> ScenarioResult:
        """Simulate random node failures with checkpoint-reload recovery."""
        total_min = training_hours * 60
        elapsed = 0.0
        num_failures = 0
        lost_progress = 0.0
        total_recovery = 0.0
        events: list[str] = []

        while elapsed < total_min:
            ttf = self._time_to_next_failure_min()
            if elapsed + ttf >= total_min:
                break
            elapsed += ttf
            failed_node = self._rng.randint(0, self.num_nodes - 1)
            num_failures += 1
            ckpt_time = self._last_checkpoint_before(elapsed)
            progress_lost = elapsed - ckpt_time
            recovery_time = self._rng.uniform(1, 5)
            lost_progress += progress_lost
            total_recovery += recovery_time
            events.append(
                f"t={elapsed:.1f}min: node {failed_node} failed; "
                f"rolled back {progress_lost:.1f}min; restart in {recovery_time:.1f}min"
            )
            elapsed += recovery_time

        survived = elapsed >= total_min or num_failures == 0
        return ScenarioResult(
            scenario="node_failure",
            survived=survived,
            lost_progress_min=round(lost_progress, 2),
            recovery_time_min=round(total_recovery, 2),
            num_failures=num_failures,
            final_step=int(elapsed),
            events=events[-5:],
        )

    def _run_gradient_corruption(self, training_hours: float) -> ScenarioResult:
        """Simulate NaN gradient injection at random steps."""
        total_min = training_hours * 60
        steps_per_min = 10
        total_steps = int(total_min * steps_per_min)
        num_failures = 0
        lost_progress = 0.0
        events: list[str] = []

        for step in range(total_steps):
            if self._rng.random() < self.failure_rate / (steps_per_min * 60):
                num_failures += 1
                lost_progress += 1 / steps_per_min
                events.append(f"step={step}: NaN gradient injected — step skipped")

        return ScenarioResult(
            scenario="gradient_corruption",
            survived=True,
            lost_progress_min=round(lost_progress, 2),
            recovery_time_min=0.0,
            num_failures=num_failures,
            final_step=total_steps,
            events=events[-5:],
        )

    def _run_slow_node(self, training_hours: float) -> ScenarioResult:
        """Simulate straggler nodes delaying the AllReduce barrier."""
        total_min = training_hours * 60
        straggler_timeout_min = 0.5
        elapsed = 0.0
        num_failures = 0
        lost_progress = 0.0
        total_recovery = 0.0
        events: list[str] = []
        active_nodes = self.num_nodes

        while elapsed < total_min:
            ttf = self._time_to_next_failure_min() * 3
            if elapsed + ttf >= total_min:
                break
            elapsed += ttf
            straggler = self._rng.randint(0, active_nodes - 1)
            delay = self._rng.uniform(straggler_timeout_min, straggler_timeout_min * 4)
            if delay > straggler_timeout_min:
                active_nodes = max(active_nodes - 1, 1)
                lost_progress += delay
                total_recovery += straggler_timeout_min
                num_failures += 1
                events.append(
                    f"t={elapsed:.1f}min: node {straggler} straggling "
                    f"({delay:.1f}min); excluded — {active_nodes} nodes remaining"
                )
            elapsed += delay

        survived = active_nodes >= self.num_nodes // 2
        return ScenarioResult(
            scenario="slow_node",
            survived=survived,
            lost_progress_min=round(lost_progress, 2),
            recovery_time_min=round(total_recovery, 2),
            num_failures=num_failures,
            final_step=int(elapsed),
            events=events[-5:],
        )

    # ------------------------------------------------------------------ public API

    def run_scenario(self, scenario: str, training_hours: float) -> ScenarioResult:
        """Run a named chaos scenario and return a structured result.

        Args:
            scenario: One of 'node_failure', 'gradient_corruption', 'slow_node'.
            training_hours: Simulated training duration in hours.

        Returns:
            ScenarioResult dataclass instance.

        Raises:
            ValueError: If scenario is not recognised.
        """
        runners = {
            "node_failure": self._run_node_failure,
            "gradient_corruption": self._run_gradient_corruption,
            "slow_node": self._run_slow_node,
        }
        if scenario not in runners:
            raise ValueError(
                f"Unknown scenario '{scenario}'. Choose from: {list(runners)}"
            )
        result = runners[scenario](training_hours)
        log.info(
            "Scenario %-24s  survived=%-5s  failures=%3d  "
            "lost=%.1f min  recovery=%.1f min",
            repr(scenario),
            result.survived,
            result.num_failures,
            result.lost_progress_min,
            result.recovery_time_min,
        )
        return result

    def monte_carlo_survival(self, trials: int) -> float:
        """Estimate job-survival probability via Monte Carlo simulation.

        Each trial simulates a full training run under node_failure chaos and
        records whether the job completes without unrecoverable progress loss.
        Bamboo elastic training: Yang et al. (2023) https://arxiv.org/abs/2303.02399

        Args:
            trials: Number of independent Monte Carlo trials.

        Returns:
            Fraction of trials in which the job survived (float in [0, 1]).
        """
        log.info("Running %d Monte Carlo trials...", trials)
        successes = 0
        for i in range(trials):
            trial_sim = DistributedTrainingSimulator(
                num_nodes=self.num_nodes,
                failure_rate=self.failure_rate,
                checkpoint_interval_min=self.checkpoint_interval_min,
                seed=i,
            )
            if trial_sim._run_node_failure(CFG["training_hours"]).survived:
                successes += 1
        prob = successes / trials
        log.info(
            "Monte Carlo survival probability: %.2f%% (%d/%d)",
            prob * 100, successes, trials,
        )
        return prob


sim = DistributedTrainingSimulator(
    num_nodes=CFG["num_nodes"],
    failure_rate=CFG["failure_rate_per_hour"],
    checkpoint_interval_min=CFG["checkpoint_interval_min"],
)


In [ ]:
# Run all chaos scenarios + Monte Carlo reliability analysis
# Check-N-Run: Eisenman et al. (2022) https://arxiv.org/abs/2401.06172

scenario_results: list[ScenarioResult] = []
for scenario in CFG["scenarios"]:
    result = sim.run_scenario(scenario, CFG["training_hours"])
    scenario_results.append(result)
    if not os.getenv("WANDB_DISABLED"):
        wandb.log({
            f"{scenario}/survived": int(result.survived),
            f"{scenario}/failures": result.num_failures,
            f"{scenario}/lost_progress_min": result.lost_progress_min,
        })
    for event in result.events:
        log.info("  Event: %s", event)

# Monte Carlo job survival probability
survival_prob = sim.monte_carlo_survival(CFG["monte_carlo_trials"])

# Reliability curve: sweep fleet sizes
try:
    import plotly.graph_objects as go  # type: ignore

    fleet_sizes = [8, 16, 32, 64, 128, 256]
    probs: list[float] = []
    for fleet_size in fleet_sizes:
        fleet_sim = DistributedTrainingSimulator(
            num_nodes=fleet_size,
            failure_rate=CFG["failure_rate_per_hour"],
            checkpoint_interval_min=CFG["checkpoint_interval_min"],
        )
        probs.append(fleet_sim.monte_carlo_survival(trials=200))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fleet_sizes,
        y=[p * 100 for p in probs],
        mode="lines+markers",
        name="Survival probability",
        line=dict(color="royalblue", width=2),
        marker=dict(size=8),
    ))
    fig.add_hline(y=95, line_dash="dash", line_color="red", annotation_text="95% SLO")
    fig.update_layout(
        title="Job Survival Probability vs Fleet Size",
        xaxis_title="Fleet size (number of nodes)",
        yaxis_title="Survival probability (%)",
        yaxis=dict(range=[0, 105]),
        xaxis=dict(type="log"),
    )
    fig.show()

    if not os.getenv("WANDB_DISABLED"):
        wandb.log({
            "survival_probability": survival_prob,
            "reliability_curve": wandb.Html(fig.to_html()),
        })

except ImportError as exc:
    log.warning("plotly not installed — skipping reliability curve: %s", exc)
    if not os.getenv("WANDB_DISABLED"):
        wandb.log({"survival_probability": survival_prob})

if not os.getenv("WANDB_DISABLED"):
    wandb.finish()
